In [3]:
import os
import math
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from torchmetrics.text import BLEUScore

from xlstm import (
    xLSTMBlockStack, xLSTMBlockStackConfig,
    mLSTMBlockConfig, mLSTMLayerConfig,
    sLSTMBlockConfig, sLSTMLayerConfig
)

class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEQ_LEN = 64
    BATCH_SIZE = 32
    EMBED_DIM = 256
    HIDDEN_DIM = 256
    LAYERS = 2
    EPOCHS = 2
    LR = 2e-3
    VOCAB_SIZE = 4000
    DATA_PATH = "./data_lm/wiki.train.raw"
    TOKENIZER_TYPE = "bpe" # 'bpe', 'wordpiece', 'unigram'

print(f"{Config.DEVICE} ")

AssertionError: c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\distutils\core.py

Corpus "abab".

1. To Bytes: Current Sequence: [97, 98, 97, 98] Vocabulary: [0...255]


2. Count Pairs 

Scan the list of bytes 97, 98, 97, 98 for adjacent pairs.

97 + 98: 2 times.

98 + 97: 1 time.

Gready approach: The pair (97, 98) is the most frequent. 

3. Merge

New Rule: (97, 98) $\rightarrow$ 256

New Sequence: [256, 256]

But do we stop?
YOU DECIDE! That rule is usually a specific Vocabulary Size.

1. Count Pairs
256 + 256: Appears 1 time.

Winner: The pair (256, 256).

2. Merge

New Rule: (256, 256) $\rightarrow$ 257

New Sequence: [257]

# WordPIECE

Corpus: "abab" (1 time)

Characters: a, b, a, b

Step 1: Pre-tokenization (Add ##)
WordPiece marks characters that do not start a word with ##.

Sequence: a, ##b, ##a, ##b

Token Counts:

a (Start): 1

##b (Inside): 2

##a (Inside): 1

Step 2: Calculate Scores

$\text{Score} = \frac{\text{Count(Pair)}}{\text{Count(Left)} \times \text{Count(Right)}}$

Count(left): Anzahl der Vorkommen des linken Tokens.

Count(right): Anzahl der Vorkommen des rechten Tokens.

Beide werden aus der aktuellen Tokenliste gezählt, bevor ein Merge ausgeführt wird.

Pair 1: a + ##b

Count(Left) = 1 (Only at the very start).
Count(Right) = 2

Math: $1 / (1 \times 2) = 0.5$

Pair 2: ##b + ##a

Count(Left) = 2.

Count(Right) = 1.

$1 / (2 \times 1) = 0.5$


Pair 3: ##a + ##b 

Left (##a): 1 timeRight (##b): 2 timesScore: $1 / (1 \times 2) = \mathbf{0.5}$

Merge: a + ##b $\rightarrow$ ab


Now Iteration 2 of wordpiece

We replace the first pair with our new token.

New Sequence: ab, ##a, ##b

New Counts:ab: 1##a: 1##b: 1 (Note: The count dropped because the other ##b got eaten!)

Calculate Scores (Iteration 2)

Now we look at the remaining connection: ##a + ##b.

Pair: ##a + ##bTogether: 1 timeLeft (##a): 1 time

Right (##b): 1 timeScore: $1 / (1 \times 1) = \mathbf{1.0}$

Vocabulary: [a, ##b, ##a, ab, ##ab]

# UNIGRAM
Corpus: "abab" 

Unigram starts by generating every possible substring that appears in the text.

Initial Vocabulary:

a

b

ab

ba

aba

bab

abab

Total Count: Sum of frequencies of all tokens.


a: 20%

b: 20%

ab: 50% 

abab: 10%

ba: 5%

aba: 5%

## Funktionsweise

Initiales Vokabular erzeugen
Alle Substrings, die im Text vorkommen (z. B. a, b, ab, aba, abab).

Wahrscheinlichkeiten schätzen
Für jedes Token wird geschätzt, wie wahrscheinlich es zur Zerlegung eines Textes beiträgt.

EM-Algorithmus (Expectation-Maximization)

E-Schritt: Finde für jeden Satz die wahrscheinlichste Zerlegung in Tokens.

M-Schritt: Berechne neue Token-Wahrscheinlichkeiten basierend auf diesen Zerlegungen.

Pruning (Reduzieren des Vokabulars)
Tokens, die die Modellwahrscheinlichkeit kaum verbessern, werden entfernt.

Finale Tokenisierung
Ein Text wird durch die Tokenfolge zerlegt, die global die höchste Wahrscheinlichkeit hat.

### Eigenschaften

Nicht deterministisch wie BPE, sondern probabilistisch.

Erlaubt mehrere Zerlegungen und wählt die beste.

Funktioniert gut bei morphologisch reichen Sprachen.

Grundlage für SentencePiece (Google) und viele moderne Modelle.

In [4]:
class DataPipeline:
    @staticmethod
    def download_wikitext():
        os.makedirs(os.path.dirname(Config.DATA_PATH), exist_ok=True)
        if not os.path.exists(Config.DATA_PATH):
            url = "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt"
            print(f"Downloading WikiText-2 from {url}...")
            r = requests.get(url)
            with open(Config.DATA_PATH, 'wb') as f: f.write(r.content)
        else:
            print("WikiText-2 already downloaded.")

    @staticmethod
    def train_tokenizer(algo_type):
        print(f"--- Training {algo_type.upper()} Tokenizer ---")
        if algo_type == 'bpe':
            tok = Tokenizer(models.BPE(unk_token="[UNK]"))
            trainer = trainers.BpeTrainer(vocab_size=Config.VOCAB_SIZE, special_tokens=["[UNK]", "[PAD]", "[EOS]"])
            tok.decoder = decoders.ByteLevel()
        elif algo_type == 'wordpiece':
            tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
            trainer = trainers.WordPieceTrainer(vocab_size=Config.VOCAB_SIZE, special_tokens=["[UNK]", "[PAD]", "[EOS]"])
            tok.decoder = decoders.WordPiece()
        elif algo_type == 'unigram':
            tok = Tokenizer(models.Unigram())
            trainer = trainers.UnigramTrainer(vocab_size=Config.VOCAB_SIZE, unk_token="[UNK]", special_tokens=["[UNK]", "[PAD]", "[EOS]"])

        tok.pre_tokenizer = pre_tokenizers.Whitespace()
        tok.train([Config.DATA_PATH], trainer)
        return tok

DataPipeline.download_wikitext()
tokenizer = DataPipeline.train_tokenizer(Config.TOKENIZER_TYPE)
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer Ready. Vocab Size: {vocab_size}")

NameError: name 'Config' is not defined

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

To predict the next token, we need chunks of size 3 (2 inputs + 1 target).
The code reshapes the list into two rows: [1, 2, 3] and [4, 5, 6].

Row 1: Input X takes [:-1] $\rightarrow$ [1, 2]. Target Y takes [1:] $\rightarrow$ [2, 3].

Row 2: Input X takes [:-1] $\rightarrow$ [4, 5]. Target Y takes [1:] $\rightarrow$ [5, 6].

Training Logic: When the model sees 1, it learns to predict 2. When it sees 1, 2, it predicts 3.

In [49]:
def get_dataloader(tokenizer):
    with open(Config.DATA_PATH, 'r', encoding='utf-8') as f: raw = f.read()
    ids = tokenizer.encode(raw).ids
    data = torch.tensor(ids, dtype=torch.long)
    n_chunks = len(data) // (Config.SEQ_LEN + 1)
    data = data[:n_chunks * (Config.SEQ_LEN + 1)].reshape(n_chunks, Config.SEQ_LEN + 1)

    class LMDataset(Dataset):
        def __len__(self): return len(data)
        def __getitem__(self, i): return data[i, :-1], data[i, 1:]

    return DataLoader(LMDataset(), batch_size=Config.BATCH_SIZE, shuffle=True)

loader = get_dataloader(tokenizer)
print(f"DataLoader Ready. Batches: {len(loader)}")

DataLoader Ready. Batches: 1417


# Autoregressives Language Modeling (LSTM / xLSTM)

## Ziel
Das Modell soll **für jedes Token in einer Sequenz das nächste Token vorhersagen**.  
Dies ist die Basis für Textgenerierung und moderne Sprachmodelle.

---

## Training – Schrittweise

1. Trainingssequenz:  
```
x = [w1, w2, w3, w4, w5]
```

2. Input & Target:

| Input (`x_in`) | Target (`y`) |
|----------------|--------------|
| w1             | w2           |
| w2             | w3           |
| w3             | w4           |
| w4             | w5           |

3. **CrossEntropyLoss** über alle Tokens:

```python
loss = CrossEntropyLoss(logits.reshape(-1, vocab_size), y.reshape(-1))
```

- Berechnet, wie gut das Modell **jeden nächsten Token** vorhersagt.
- Teacher Forcing: Modell bekommt **immer die echten Tokens** als Input.

---

## Autoregressives Sampling (Generierung)

1. Start-Prompt: `[w1]`  
2. Modell sagt `w2_pred` voraus  
3. `w2_pred` wird wieder als Input genutzt → Modell sagt `w3_pred` voraus  
4. Wiederhole, bis gewünschte Länge erreicht ist

```
w1 -> w2_pred -> w3_pred -> w4_pred -> ...
```

- Hier gibt es **keine echten Targets**, nur generierte Tokens.
- `hidden state` wird bei Standard-LSTM genutzt, beim xLSTM ist er optional.

---

## Unterschied Training vs. Generierung

| Phase       | Input                    | Target                | Bemerkung |
|------------|--------------------------|----------------------|-----------|
| Training   | echte Tokens             | echte nächste Tokens | Modell lernt Regeln der Sprache, Fehler werden korrigiert |
| Generierung   | bisher generierte Tokens | unbekannt            | Modell wendet Gelerntes an, Text wird erzeugt |

---

## Wichtige Punkte

- Training: **parallel über ganze Sequenzen**  
- Sampling: **Token für Token autoregressiv**  
- Ziel: **nächstes Token vorhersagen**, nicht ganze Sequenz gleichzeitig


In [50]:
class StandardLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, Config.EMBED_DIM)
        self.lstm = nn.LSTM(Config.EMBED_DIM, Config.HIDDEN_DIM, Config.LAYERS, batch_first=True)
        self.head = nn.Linear(Config.HIDDEN_DIM, vocab_size)

    def forward(self, x, hidden=None):
        x = self.emb(x)
        out, hidden = self.lstm(x, hidden)
        return self.head(out), hidden

class ExtendedLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        cfg = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(num_heads=4, conv1d_kernel_size=4)),
            slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(num_heads=4)),
            num_blocks=Config.LAYERS,
            embedding_dim=Config.EMBED_DIM,
            context_length=Config.SEQ_LEN * 2,
            dropout=0.1
        )
        self.emb = nn.Embedding(vocab_size, Config.EMBED_DIM)
        self.xlstm = xLSTMBlockStack(cfg)
        self.head = nn.Linear(Config.EMBED_DIM, vocab_size)

    def forward(self, x, hidden=None):
        x = self.emb(x)
        out = self.xlstm(x)
        return self.head(out), None

In [ ]:
class Engine:
    @staticmethod
    def train(model, loader, vocab_size):
        model.train().to(Config.DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=Config.LR)
        loss_fn = nn.CrossEntropyLoss()

        for epoch in range(Config.EPOCHS):
            loop = tqdm(loader, desc=f"Epoch {epoch+1}")
            for x, y in loop:
                x, y = x.to(Config.DEVICE), y.to(Config.DEVICE)
                opt.zero_grad()
                logits, _ = model(x)
                loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
                loss.backward()
                opt.step()
                loop.set_description(f"Loss: {loss.item():.4f}")

    @staticmethod
    def evaluate(model, tokenizer, test_prompt="The meaning of life is"):
        model.eval().to(Config.DEVICE)
        ids = tokenizer.encode(test_prompt).ids
        if len(ids) < 2: ids = ids + [0] 
        x_in = torch.tensor([ids[:-1]]).to(Config.DEVICE)
        y_true = torch.tensor([ids[1:]]).to(Config.DEVICE)

        with torch.no_grad():
            logits, _ = model(x_in)
            loss = F.cross_entropy(logits.reshape(-1, tokenizer.get_vocab_size()), y_true.reshape(-1))
            ppl = math.exp(loss.item())

        gen_ids = list(ids)
        curr_in = torch.tensor([ids]).to(Config.DEVICE)
        hidden = None

        with torch.no_grad():
            for _ in range(30): 
                logits, hidden = model(curr_in, hidden)
                next_token = torch.argmax(logits[:, -1, :], dim=-1) #einfach das Wort mit höchster Wahrscheinlihckeit´, kein sampling
                gen_ids.append(next_token.item())
                curr_in = torch.cat([curr_in, next_token.unsqueeze(0)], dim=1)
                if curr_in.shape[1] > Config.SEQ_LEN: curr_in = curr_in[:, -Config.SEQ_LEN:]

        gen_text = tokenizer.decode(gen_ids)
        bleu = BLEUScore()([gen_text], [[test_prompt + " something something"]]).item()

        return ppl, bleu, gen_text

In [52]:
models_dict = {
    "LSTM": StandardLSTM(vocab_size),
    "xLSTM": ExtendedLSTM(vocab_size)
}

results = {}
print("\n--- Starting Training Comparison ---")

for name, model in models_dict.items():
    print(f"\nTraining {name}...")
    Engine.train(model, loader, vocab_size)
    ppl, bleu, text = Engine.evaluate(model, tokenizer)
    results[name] = {"PPL": ppl, "BLEU": bleu, "Text": text}


--- Starting Training Comparison ---

Training LSTM...


Loss: 4.4584: 100%|██████████| 1417/1417 [00:20<00:00, 70.56it/s]



Training xLSTM...


Loss: 3.9543: 100%|██████████| 1417/1417 [00:37<00:00, 38.08it/s]


In [53]:
print("\n" + "="*60)
print(f"{'FINAL LEADERBOARD':^60}")
print("="*60)
print(f"{'Model':<10} | {'Perplexity (Lower is better)':<30} | {'BLEU':<10}")
print("-" * 60)
for name, res in results.items():
    print(f"{name:<10} | {res['PPL']:<30.2f} | {res['BLEU']:<10.4f}")
print("-" * 60)
print(f"LSTM Gen : {results['LSTM']['Text']}")
print(f"xLSTM Gen: {results['xLSTM']['Text']}")


                     FINAL LEADERBOARD                      
Model      | Perplexity (Lower is better)   | BLEU      
------------------------------------------------------------
LSTM       | 155.68                         | 0.0000    
xLSTM      | 177.49                         | 0.0000    
------------------------------------------------------------
LSTM Gen : Themeaningoflifeisa<unk>,andthe<unk><unk>,and<unk>,<unk>,<unk>,<unk>
xLSTM Gen: Themeaningoflifeisa<unk>,whichisa<unk>,whichisa<unk><unk><unk><unk><unk>


Perplexity

If you are 100% sure the next word is "cat", your perplexity is 1. (Perfect).If you are torn between 2 words ("cat" or "dog"), your perplexity is 2.If you have no idea and are guessing from 100 words, your perplexity is 100.The Math: It is calculated as $e^{\text{CrossEntropyLoss}}$.

BLUE

Reference: "The quick brown fox."

Model A: "The quick brown dog."

Model B: "A fast dark fox."

How BLEU sees it:

Model A: It got the 2-gram chunk "quick brown" exactly right. High Score.

Model B: It got 0 chunks right (even though the meaning is correct!). Score 0.

# Evaluationsmetriken für Sprachmodelle

## 1. Perplexity (PPL)

Perplexity misst, **wie gut ein Sprachmodell Vorhersagen trifft**.  
Je niedriger die Perplexity, desto besser ist das Modell darin, die tatsächlichen Tokens vorherzusagen.

### Definition

Für eine Sequenz von Tokens \(x_1, x_2, ..., x_T\) und ein Modell, das Wahrscheinlichkeiten \(P(x_t|x_1,...,x_{t-1})\) liefert:

$$
\text{PPL} = \exp \Bigg( - \frac{1}{T} \sum_{t=1}^{T} \log P(x_t | x_1,...,x_{t-1}) \Bigg)
$$

### Intuition

- Perplexity = „durchschnittliche Anzahl der Optionen, die das Modell pro Token hat“  
- Beispiel: PPL = 10 → im Schnitt denkt das Modell wie aus 10 gleich wahrscheinlichen Tokens gewählt wird.  
- Niedrigere PPL → Modell ist **sicherer und genauer** bei Token-Vorhersagen.

### ⚠️ Wichtig: Sicher ≠ richtig

| Token | Modellvorhersage | Wahrscheinlichkeit des richtigen Tokens | Kommentar |
|-------|-----------------|---------------------------------------|-----------|
| "cat" | "cat"           | 0.99                                  | PPL niedrig, korrekt |
| "cat" | "dog"           | 0.01                                  | PPL hoch, unsicher oder falsch |
| "cat" | "dog"           | 0.99 für "dog", richtige "cat" = 0.01 | Modell sehr sicher, **aber falsch** → PPL hoch, misst nur Unsicherheit bzgl. richtiger Tokens |

**Fazit:**  
- PPL misst die Unsicherheit des Modells in Bezug auf die **richtigen Trainings-Tokens**  
- Eine niedrige PPL zeigt, dass das Modell die Verteilung gelernt hat, garantiert aber nicht immer korrekte generierte Tokens.

---

## 2. BLEU-Score

BLEU (Bilingual Evaluation Understudy) misst, **wie ähnlich generierter Text zu Referenztexten ist**.  
Häufig verwendet für **Maschinenübersetzung oder Textgenerierung**.

### Berechnung

1. Zerlege Text in n-Gramme (z.B. Bi-, Tri-Gramme)  
2. Zähle, wie viele n-Gramme im generierten Text mit den Referenzen übereinstimmen  
3. Korrigiere mit einem **Brevity Penalty**, falls der generierte Text kürzer ist  

$$
\text{BLEU} = BP \cdot \exp \left( \sum_{n=1}^{N} w_n \log p_n \right)
$$

- \(p_n\) = Präzision der n-Gramme  
- \(w_n\) = Gewichtung für verschiedene n-Gramme  
- \(BP\) = Strafe für zu kurzen Text

### Intuition

- BLEU = 1 (oder 100%) → perfekte Übereinstimmung mit Referenz  
- BLEU = 0 → keine Übereinstimmung  
- Misst **nicht die Semantik**, nur die Formulierung und Übereinstimmung der Wortfolgen.

---

## 🔹 Zusammenfassung

| Metrik      | Was misst sie?                       | Ziel |
|------------|-------------------------------------|------|
| Perplexity | Vorhersagewahrscheinlichkeit pro Token | Niedrig → besser, misst Sicherheit bzgl. echten Tokens |
| BLEU       | Übereinstimmung von n-Grammen mit Referenztext | Hoch → besser, misst Qualität generierter Texte |

**Hinweis:**  
- PPL eignet sich **für Training & Evaluation**, weil sie direkt aus den Logits berechnet wird.  
- BLEU eignet sich **für generierten Text**, um die Qualität im Vergleich zu einem Referenztext zu beurteilen.
